<h2> Feature selection - Chi and Anova </h2>

In [19]:
from sklearn.model_selection import train_test_split
import pandas as pd

df_normalized = pd.read_csv('normalized_vaccine_data_processed.csv') # change csv file name
target_normalized = df_normalized['AgreeSubsequentBooster']


features_normalized = df_normalized.drop('AgreeSubsequentBooster', axis=1)

x_train_normalized, x_test_normalized, y_train_normalized, y_test_normalized = train_test_split(features_normalized,
                                                                                                target_normalized,
                                                                                                test_size=0.2,
                                                                                                random_state=0)

In [20]:
anova_columns = [
    'Age','EmploymentYears',

    'S.VaccineImportantHealth','S.VaccineEffectiveSevere','S.VaccineImportantPublic',
    'S.VaccineBeneficialMOH','S.VaccineSafe','S.MOHInfoTrusted',
    'S.VaccineProtectInfection','S.AgreeMOHRecommendation','S.WorriedSideEffects',
    'S.PreventNewVariant','P.ProtectFlu','P.ProtectSevereComp','P.EffectivenessVsRisk',
    'P.ClinicallyTested','P.LongTermSideEffects','P.TrustReportsMOH','P.ChipBelief',
    'P.MandatoryBelief','P.HalalDoubt','P.AlternativeMedicineBelief','P.HighRiskJob',
    'P.WorkDemand','A.BoosterProtectSevere','A.BoosterProtectFamily',
    'A.BoosterPreventSpread','A.ReturnDailyActivities','A.RecommendingBooster',
    'A.MaintainAntibody','A.NoSeriousSideEffects','B.RiskWithoutBooster',
    'B.SevereCompWithoutBooster','B.SpreadFamilyWithoutBooster',
    'B.TransmitPatientsWithoutBooster','B.SkepticalBooster','B.BoosterSafe',
    'B.BoosterMoreSideEffects','B.PreferNaturalImmunity','B.EliminateStigma',
    'B.PublicRiskWithoutBooster','C.BoosterNotNeeded','C.BoosterHarm',
    'C.EvidenceInsufficient','C.TrustSocialMedia','C.SideEffectsNoImpact'
]

chi2_columns = [
    'Gender','Education','Income','Pregnant','PatientContact','Occupation',
    'CovidPatientCare','PreExistingCondition','CovidInfected','Severity',
    'AdditionalVaccines','VaccinationStage','VaccinationSideEffects',
    'FamilySideEffects','SideEffectsAffectView',

    # one-hot race dummies
    'Race_Chinese','Race_Indian','Race_Malay','Race_Siam',

    # one-hot religion dummies
    'Religion_Buddha','Religion_Hindu','Religion_Islam',

    # one-hot location dummies
    'Location_Johor','Location_Kedah','Location_Kelantan','Location_Kuantan ',
    'Location_Melaka','Location_Negeri Sembilan','Location_Pahang',
    'Location_Perlis','Location_Pulau Pinang','Location_Sabah',
    'Location_Selangor','Location_Terengganu'
]


# Extract and save ANOVA columns from the training set
anova_train = x_train_normalized[anova_columns]

# Extract and save Chi-Squared columns from the training set
chi2_train = x_train_normalized[chi2_columns]

<h2>Anova testing</h2>

In [21]:
import pandas as pd
# Import SelectKBest and chi2 modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif


# Apply SelectKBest with ANOVA on the anova_train data
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(anova_train, y_train_normalized)
scores = selector.scores_
p_values = selector.pvalues_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': anova_train.columns,
    'ANOVA Score': scores,
    'P-value': p_values
})

# Sort the DataFrame by ANOVA scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='ANOVA Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                             Feature  ANOVA Score       P-value
11               S.PreventNewVariant    72.812733  9.375199e-16
28             A.RecommendingBooster    69.486772  3.587994e-15
26            A.BoosterPreventSpread    67.829306  7.039559e-15
29                A.MaintainAntibody    63.403741  4.329973e-14
25            A.BoosterProtectFamily    63.354303  4.419370e-14
5             S.VaccineBeneficialMOH    59.552243  2.148391e-13
24            A.BoosterProtectSevere    57.113772  5.984172e-13
34  B.TransmitPatientsWithoutBooster    56.442802  7.944030e-13
2           S.VaccineImportantHealth    54.179077  2.075641e-12
33      B.SpreadFamilyWithoutBooster    51.921263  5.448918e-12
31              B.RiskWithoutBooster    51.154863  7.573815e-12
8          S.VaccineProtectInfection    49.649681  1.449620e-11
40        B.PublicRiskWithoutBooster    47.661615  3.434614e-11
9           S.AgreeMOHRecommendation    45.881570  7.473101e-11
4           S.VaccineImportantPublic    

In [22]:
# Select features with p-value < 0.05 # 0.05 is the default value. less than 0.05 is better
significant_features_anova = feature_scores[feature_scores['P-value'] < 0.05]

# Display significant features
print(significant_features_anova)

                             Feature  ANOVA Score       P-value
1                    EmploymentYears     8.079989  4.807366e-03
2           S.VaccineImportantHealth    54.179077  2.075641e-12
3           S.VaccineEffectiveSevere    45.346041  9.451222e-11
4           S.VaccineImportantPublic    45.427927  9.117611e-11
5             S.VaccineBeneficialMOH    59.552243  2.148391e-13
6                      S.VaccineSafe    43.366650  2.260119e-10
7                   S.MOHInfoTrusted    40.810196  7.033411e-10
8          S.VaccineProtectInfection    49.649681  1.449620e-11
9           S.AgreeMOHRecommendation    45.881570  7.473101e-11
11               S.PreventNewVariant    72.812733  9.375199e-16
13               P.ProtectSevereComp    29.266737  1.361452e-07
14             P.EffectivenessVsRisk    32.747922  2.712208e-08
15                P.ClinicallyTested    30.094555  9.256833e-08
16             P.LongTermSideEffects     5.145617  2.407214e-02
17                 P.TrustReportsMOH    

<h2>Chi2 Testing</h2>

In [23]:
# Import SelectKBest and chi2 modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2


# Apply SelectKBest with chi2
selector = SelectKBest(score_func=chi2, k='all')
selector.fit(chi2_train, y_train_normalized)
scores = selector.scores_
p_values = selector.pvalues_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': chi2_train.columns,
    'Chi-squared Score': scores,
    'P-value': p_values
})

# Sort the DataFrame by Chi-squared scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='Chi-squared Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                     Feature  Chi-squared Score       P-value
14     SideEffectsAffectView          32.492457  1.196561e-08
1                  Education          11.733441  6.138700e-04
6           CovidPatientCare           5.529605  1.869732e-02
33       Location_Terengganu           5.396478  2.017743e-02
15              Race_Chinese           5.052632  2.458857e-02
2                     Income           4.211019  4.016220e-02
19           Religion_Buddha           2.786184  9.508039e-02
32         Location_Selangor           2.526316  1.119614e-01
30     Location_Pulau Pinang           2.526316  1.119614e-01
20            Religion_Hindu           2.526316  1.119614e-01
16               Race_Indian           2.526316  1.119614e-01
0                     Gender           2.006638  1.566120e-01
22            Location_Johor           1.684211  1.943659e-01
11          VaccinationStage           1.551639  2.128937e-01
9                   Severity           1.546160  2.137032e-01
18      

In [24]:
# Select features with p-value < 0.05
significant_features_chi = feature_scores[feature_scores['P-value'] < 0.05]

# Display significant features
print(significant_features_chi)

                  Feature  Chi-squared Score       P-value
1               Education          11.733441  6.138700e-04
2                  Income           4.211019  4.016220e-02
6        CovidPatientCare           5.529605  1.869732e-02
14  SideEffectsAffectView          32.492457  1.196561e-08
15           Race_Chinese           5.052632  2.458857e-02
33    Location_Terengganu           5.396478  2.017743e-02


<h2>Information Gain</h2>

In [25]:
# Import SelectKBest and information gain modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

# Apply SelectKBest with information gain
selector = SelectKBest(score_func=mutual_info_classif, k='all')
selector.fit(chi2_train, y_train_normalized)
scores = selector.scores_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': chi2_train.columns,
    'Information gain Score': scores,
})

# Sort the DataFrame by Chi-squared scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='Information gain Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                     Feature  Information gain Score
1                  Education                0.046996
16               Race_Indian                0.043614
14     SideEffectsAffectView                0.040056
7       PreExistingCondition                0.032361
33       Location_Terengganu                0.031588
2                     Income                0.031245
5                 Occupation                0.027657
17                Race_Malay                0.026364
20            Religion_Hindu                0.025700
3                   Pregnant                0.018003
21            Religion_Islam                0.016616
12    VaccinationSideEffects                0.014332
4             PatientContact                0.011031
8              CovidInfected                0.010422
32         Location_Selangor                0.009850
19           Religion_Buddha                0.008973
10        AdditionalVaccines                0.007853
6           CovidPatientCare                0.

In [26]:
selected_features_in = feature_scores_sorted[feature_scores_sorted['Information gain Score'] > 0.05]
print(selected_features_in)

Empty DataFrame
Columns: [Feature, Information gain Score]
Index: []


<h2>Wrapper approach</h2>

<h2>RFE</h2>

In [27]:
# Import necessary libraries
from sklearn.feature_selection import RFE
from sklearn.svm import SVC

# Create an SVM classifier with a linear kernel
svm_linear = SVC(kernel='linear')

# Initialize RFE with the SVM classifier
rfe = RFE(estimator=svm_linear, step=1)

# Fit RFE on the training data
rfe.fit(x_train_normalized, y_train_normalized)

# Print the results
print("Number of Features: ", rfe.n_features_)
print("Feature Ranking: ", rfe.ranking_)
print("Selected Features: ", rfe.support_)

# Print the indices of the selected features
selected_feature_indices = [i for i, x in enumerate(rfe.support_) if x]
print("Selected Feature Indices: ", selected_feature_indices)

Number of Features:  41
Feature Ranking:  [ 1  1  1 18 11  1 34 30  1 25  1 28  1  1  1 27  1  1 31  1  1  1  3  1
 17  5  1 12  1  1  1  1 24  1 36  1 22  1 32  1  1  1  1 14 23  1 26  9
  8 33  1 29 16  1  1 21  1  1  1  1 20  4  2  1 41  6 13  1 42  7 19 35
 40  1 37 15 38 39  1  1  1 10]
Selected Features:  [ True  True  True False False  True False False  True False  True False
  True  True  True False  True  True False  True  True  True False  True
 False False  True False  True  True  True  True False  True False  True
 False  True False  True  True  True  True False False  True False False
 False False  True False False  True  True False  True  True  True  True
 False False False  True False False False  True False False False False
 False  True False False False False  True  True  True False]
Selected Feature Indices:  [0, 1, 2, 5, 8, 10, 12, 13, 14, 16, 17, 19, 20, 21, 23, 26, 28, 29, 30, 31, 33, 35, 37, 39, 40, 41, 42, 45, 50, 53, 54, 56, 57, 58, 59, 63, 67, 73, 78, 79, 80]


In [28]:
# Get the names of the selected features
selected_feature_names = x_train_normalized.columns[rfe.support_]
print("Selected Feature Names:")
for feature_name in selected_feature_names.tolist():
    print(feature_name)

Selected Feature Names:
Age
Gender
Education
PatientContact
EmploymentYears
CovidInfected
AdditionalVaccines
VaccinationStage
VaccinationSideEffects
SideEffectsAffectView
S.VaccineImportantHealth
S.VaccineImportantPublic
S.VaccineBeneficialMOH
S.VaccineSafe
S.VaccineProtectInfection
S.PreventNewVariant
P.ProtectSevereComp
P.EffectivenessVsRisk
P.ClinicallyTested
P.LongTermSideEffects
P.ChipBelief
P.HalalDoubt
P.HighRiskJob
A.BoosterProtectSevere
A.BoosterProtectFamily
A.BoosterPreventSpread
A.ReturnDailyActivities
A.NoSeriousSideEffects
B.SkepticalBooster
B.PreferNaturalImmunity
B.EliminateStigma
C.BoosterNotNeeded
C.BoosterHarm
C.EvidenceInsufficient
C.TrustSocialMedia
Race_Malay
Religion_Islam
Location_Melaka
Location_Pulau Pinang
Location_Sabah
Location_Selangor


<h2>OUTPUT</h2>

In [29]:
result1 = pd.concat([significant_features_anova,significant_features_chi])
result1 = result1.drop(columns=['ANOVA Score', 'P-value', 'Chi-squared Score'])
print (result1)

                             Feature
1                    EmploymentYears
2           S.VaccineImportantHealth
3           S.VaccineEffectiveSevere
4           S.VaccineImportantPublic
5             S.VaccineBeneficialMOH
6                      S.VaccineSafe
7                   S.MOHInfoTrusted
8          S.VaccineProtectInfection
9           S.AgreeMOHRecommendation
11               S.PreventNewVariant
13               P.ProtectSevereComp
14             P.EffectivenessVsRisk
15                P.ClinicallyTested
16             P.LongTermSideEffects
17                 P.TrustReportsMOH
18                      P.ChipBelief
19                 P.MandatoryBelief
20                      P.HalalDoubt
21       P.AlternativeMedicineBelief
23                      P.WorkDemand
24            A.BoosterProtectSevere
25            A.BoosterProtectFamily
26            A.BoosterPreventSpread
27           A.ReturnDailyActivities
28             A.RecommendingBooster
29                A.MaintainAntibody
3

In [30]:
significant_features = result1['Feature'].tolist()
anova_chi = df_normalized[significant_features + ['AgreeSubsequentBooster']]
anova_chi.to_csv('anova_chi_normalized.csv', index=False) # 200 features
anova_chi.to_csv('ac_350_test.csv') # 350 features

In [31]:
# anova_chi_not_normalized = df_not_normalized[significant_features + ['AgreeSubsequentBooster']]
# anova_chi_not_normalized.to_csv('anova_chi_not_normalized.csv', index=False)

In [32]:
result2 = pd.concat([significant_features_anova,selected_features_in])
result2 = result2.drop(columns=['ANOVA Score', 'P-value', 'Information gain Score'])
print (result2)

In [33]:
significant_features = result2['Feature'].tolist()
anova_in = df_normalized[significant_features + ['AgreeSubsequentBooster']]
anova_in.to_csv('anova_cin_normalized.csv', index=False)

In [34]:
# anova_in_not_normalized = df_not_normalized[significant_features + ['AgreeSubsequentBooster']]
# anova_in_not_normalized.to_csv('anova_cin_not_normalized.csv', index=False)

In [35]:
result3 = pd.concat([significant_features_chi,selected_features_in])
result3 = result3.drop(columns=['Chi-squared Score', 'P-value', 'Information gain Score'])
print (result3)

In [36]:
significant_features = result3['Feature'].tolist()
chi_in = df_normalized[significant_features + ['AgreeSubsequentBooster']]
chi_in.to_csv('chi_in.csv', index=False)